In [1]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np

# 1. Building a TF-IDF vectorizer

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_review.csv")

In [3]:
df.isnull().sum()

rating                0
title                 6
text                  4
asin                  0
timestamp             0
helpful_vote          0
verified_purchase     0
text_clean           20
text_clean_full      13
dtype: int64

In [4]:
df_copy = df.copy()

### Noise removal
- **20 reviews in text_clean contained only stopwords — became empty strings after stopword removal, dropped here**,
- **13 reviews in text_clean_full became empty after preprocessing (likely very short reviews with no alphabetic content), dropped here**
- **6 empty Titles removed**
- **4 empty Text reviews removed**

In [5]:
df = df.dropna(subset=['text_clean'])
df = df.dropna(subset=['text_clean_full'])
df = df.dropna(subset=['text'])
df = df.dropna(subset=['title'])

In [6]:
df.isnull().sum()

rating               0
title                0
text                 0
asin                 0
timestamp            0
helpful_vote         0
verified_purchase    0
text_clean           0
text_clean_full      0
dtype: int64

In [7]:
df.shape

(30157, 9)

In [9]:
df.to_csv("../data/cleaned/cleaned_review.csv",index=False)

In [8]:
vectorizer = TfidfVectorizer(min_df=5,max_features=10000,token_pattern=r'[a-zA-z]{2,}')
matrix = vectorizer.fit_transform(df['text_clean'])

In [9]:
matrix.shape

(30157, 6088)

In [10]:
print(vectorizer.get_feature_names_out()[:20])

['ability' 'able' 'abs' 'absolute' 'absolutely' 'absorb' 'absorbent'
 'absorbing' 'absorbs' 'absorption' 'absurd' 'abuse' 'abused' 'abusive'
 'abysmal' 'ac' 'accent' 'accept' 'acceptable' 'accepted']


In [11]:
print(vectorizer.get_feature_names_out()[100:120])

['advance' 'advantage' 'advantages' 'advertise' 'advertised'
 'advertisement' 'advertises' 'advertising' 'advice' 'advise' 'advised'
 'ae' 'aegis' 'aesthetic' 'aesthetically' 'aesthetics' 'af' 'affect'
 'affected' 'affects']


# 2. TF-IDF Search Function

In [12]:
import re
import emoji
import nltk
from nltk.corpus import stopwords
def text_preprocessing(text,remove_stopwords=False):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text)
    text = re.sub(r'&[a-z]+;|&#\d+;', '', text) 
    text = re.sub(r'<[^>]+>', '', text) 
    text = emoji.replace_emoji(text,replace="")
    special_character_pattern = r'[^a-zA-Z0-9\s]+'
    text = re.sub(special_character_pattern,"",text)
    text = re.sub(r"\s+"," ",text).strip()
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        words = text.split(" ")
        words = [word for word in words if word not in stop_words]
        text = " ".join(words)
    
    return text

In [13]:
from sklearn.metrics.pairwise import cosine_similarity

def search(query,vectorizer,matrix,df,top_k=5):
    query = text_preprocessing(query,remove_stopwords=True)
    query_matrix = vectorizer.transform([query])
    cos_similarity = cosine_similarity(matrix,query_matrix).flatten() # to get a 1D array for argsort
    asc_indexes = np.argsort(cos_similarity)
    desc_indexes = asc_indexes[::-1]
    desc_indexes = desc_indexes[:top_k]
    return df.iloc[desc_indexes]

In [14]:
results = search("battery not charging",vectorizer,matrix,df,top_k=5)
print(results[['rating','title','text']])

       rating                                  title  \
25898     2.0                                money 💰   
10686     1.0  Does not work, don't waste your money   
29311     2.0                                  Nope.   
10968     2.0            Better think twice b4 u buy   
9609      1.0                  Charger doesnt charge   

                                                    text  
25898                                     the battery 🔋👎  
10686  Even after fully charging the battery, it cann...  
29311                             Battery does not last.  
10968  It seems like i had bought an already used bat...  
9609   Although it shows in my phone that it's chargi...  


# 3. Testing on 10 handwritten queries

In [15]:
queries = [
    "Battery not working",
    "Charging wire not working",
    "Screen is broken",
    "Packaging is broken",
    "Bluetooth not working",
    "Broken Case",
    "Incompatible accessory",
    "Bad sound quality",
    "Uncomfortable while wearing",
    "the product is not what it markets itself"
]
for q in queries:
    print(f"Query: {q}")
    results = search(q,vectorizer,matrix,df,top_k=5)
    print(results[['rating','title','text']])

Query: Battery not working
       rating                title            text
25898     2.0              money 💰  the battery 🔋👎
4729      1.0      Replacement pen   Not working ?
18432     1.0                Break     Not working
9638      1.0  Did not get to work     Not working
4644      1.0             One Star    Not working!
Query: Charging wire not working
       rating                       title  \
13174     1.0  Won't charge after a month   
7546      1.0   USB charging cable burned   
18432     1.0                       Break   
4644      1.0                    One Star   
4729      1.0             Replacement pen   

                                                    text  
13174                                 Wire not original.  
7546   When I was charging the device, the charging c...  
18432                                        Not working  
4644                                        Not working!  
4729                                       Not working ?  
Query: Sc

## TF-IDF Baseline — Evaluation Results

Tested on 10 hand-written queries covering battery, charging, screen, packaging,
Bluetooth, case, compatibility, sound, comfort, and general disappointment complaints.

**Strong performance:** Queries with rare, distinctive words ("incompatible",
"uncomfortable") — TF-IDF correctly surfaced relevant complaints.

**Weak performance:** Queries dominated by generic words ("broken", "not working",
"bad quality") — TF-IDF matched the common word but ignored context, returning
results that share vocabulary but not necessarily meaning.

**Conclusion:** TF-IDF is a fast, explainable baseline but fundamentally limited by
exact word matching. This brings Phase 3 in the picture — sentence embeddings, which capture
semantic meaning rather than literal word overlap.